# 05 | Juntar CDC e RAND para a pergunta principal

Unimos a taxa de homicídios do CDC e a contagem de registros de regras da RAND por **estado e mesmo ano**. O fluxo principal exige apenas os notebooks 01 e 02. As fontes econômicas BLS e BEA pertencem ao experimento anterior de previsão.

## Bloco 1: Abrir as tabelas

Execute os notebooks 01 e 02 antes deste. O arquivo de contagem RAND é gerado pelo bloco 3 do notebook 02.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.association_data import build_association_table
p = ROOT / "data"
cdc = pd.read_csv(p / "raw/cdc_homicides_2019_2024.csv")
contagem = pd.read_csv(p / "processed/rand_law_counts_2019_2024.csv")
print("Linhas abertas:", {"CDC": len(cdc), "RAND": len(contagem)})

## Bloco 2: Integrar pelo mesmo ano

Cada linha representa um estado em um ano. A taxa de homicídios é por 100 mil habitantes.

In [ ]:
base = build_association_table(cdc, contagem)
print("Base integrada:", base.shape)
display(base.head())

## Bloco 3: Conferir e salvar

Esperamos 50 estados em cada um dos seis anos, totalizando 300 linhas. A soma das contagens restritivas e permissivas deve ser igual à contagem total.

In [ ]:
assert base.shape == (300, 7)
assert base.groupby("year")["state"].nunique().eq(50).all()
assert not base.duplicated(["state", "year"]).any()
assert base["total_law_records"].eq(base["restrictive_records"] + base["permissive_records"]).all()
destino = p / "processed/us_gun_policy_association_2019_2024.csv"
base.to_csv(destino, index=False)
print("Base salva:", destino.relative_to(ROOT))
display(base.groupby("year").agg(estados=("state", "nunique"), menor_contagem=("total_law_records", "min"), maior_contagem=("total_law_records", "max")))